In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import pickle
import random
from tqdm import tqdm

In [ ]:
class SiameseSignatureDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
        self.transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=3),  
            transforms.ToTensor(),                        
        ])
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        img1, img2, label = self.pairs[idx]
        img1 = self.transform(img1)
        img2 = self.transform(img2)
        label = torch.tensor(label, dtype=torch.float32)
        return img1, img2, label

In [ ]:
class SiameseResNet(nn.Module):
    def __init__(self):
        super(SiameseResNet, self).__init__()
        base_model = models.resnet18(weights=None)
        base_model.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        num_ftrs = base_model.fc.in_features
        base_model.fc = nn.Identity()
        self.base_model = base_model
        self.embedding = nn.Sequential(
            nn.Linear(num_ftrs, 256),
            nn.ReLU(),
            nn.Linear(256, 128)
        )
    def forward_once(self, x):
        x = self.base_model(x)
        x = self.embedding(x)
        return x
    def forward(self, img1, img2):
        out1 = self.forward_once(img1)
        out2 = self.forward_once(img2)
        return out1, out2

In [ ]:
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin
    def forward(self, out1, out2, label):
        euclidean_distance = F.pairwise_distance(out1, out2)
        loss = label * torch.pow(euclidean_distance, 2) + \
               (1 - label) * torch.pow(torch.clamp(self.margin - euclidean_distance, min=0.0), 2)
        return torch.mean(loss)

In [ ]:
with open("C:/Active-Repositories/secure-signature-verification/image_pairs/positive_pairs.pkl", "rb") as f1:
    positive_pairs = pickle.load(f1)
with open("C:/Active-Repositories/secure-signature-verification/image_pairs/negative_pairs.pkl", "rb") as f2:
    negative_pairs = pickle.load(f2)
all_pairs = positive_pairs + negative_pairs
random.shuffle(all_pairs)
split = int(0.8 * len(all_pairs))
train_pairs = all_pairs[:split]
test_pairs = all_pairs[split:]
train_dataset = SiameseSignatureDataset(train_pairs)
test_dataset = SiameseSignatureDataset(test_pairs)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SiameseResNet().to(device)
criterion = ContrastiveLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
epochs = 5
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for img1, img2, label in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        img1, img2, label = img1.to(device), img2.to(device), label.to(device)
        out1, out2 = model(img1, img2)
        loss = criterion(out1, out2, label)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

In [ ]:
torch.save(model.state_dict(), "trained_model/siamese_resnet_18.pth")